In [1]:
import pandas as pd
import tensorflow as tf
import json
import random
import logging
import re

# Named-Entity Recognition to Process Resumes

In [2]:
df_data = pd.read_json("ner.json", lines=True)
df_data = df_data.drop(['extras'], axis=1)
df_data['content'] = df_data['content'].str.replace("\n", " ")

In [3]:
df_data.head()

,content,annotation
0,Abhishek Jha Application Development Associate...,"[{'label': ['Skills'], 'points': [{'start': 12..."
1,Afreen Jamadar Active member of IIIT Committee...,"[{'label': ['Email Address'], 'points': [{'sta..."
2,"Akhil Yadav Polemaina Hyderabad, Telangana - E...","[{'label': ['Skills'], 'points': [{'start': 37..."
3,Alok Khandai Operational Analyst (SQL DBA) Eng...,"[{'label': ['Skills'], 'points': [{'start': 80..."
4,Ananya Chavan lecturer - oracle tutorials Mum...,"[{'label': ['Degree'], 'points': [{'start': 20..."


In [4]:
df_data.iloc[0]['annotation']

[{'label': ['Skills'],
  'points': [{'start': 1295,
    'end': 1621,
    'text': '\n• Programming language: C, C++, Java\n• Oracle PeopleSoft\n• Internet Of Things\n• Machine Learning\n• Database Management System\n• Computer Networks\n• Operating System worked on: Linux, Windows, Mac\n\nNon - Technical Skills\n\n• Honest and Hard-Working\n• Tolerant and Flexible to Different Situations\n• Polite and Calm\n• Team-Player'}]},
 {'label': ['Skills'],
  'points': [{'start': 993,
    'end': 1153,
    'text': 'C (Less than 1 year), Database (Less than 1 year), Database Management (Less than 1 year),\nDatabase Management System (Less than 1 year), Java (Less than 1 year)'}]},
 {'label': ['College Name'],
  'points': [{'start': 939, 'end': 956, 'text': 'Kendriya Vidyalaya'}]},
 {'label': ['College Name'],
  'points': [{'start': 883, 'end': 904, 'text': 'Woodbine modern school'}]},
 {'label': ['Graduation Year'],
  'points': [{'start': 856, 'end': 860, 'text': '2017\n'}]},
 {'label': ['College 

In [5]:
def mergeIntervals(intervals):
    sorted_by_lower_bound = sorted(intervals, key=lambda tup:tup[0])
    merged = []

    for higher in sorted_by_lower_bound:
        if not merged:
            merged.append(higher)
        else:
            lower = merged[-1]
            if higher[0] <= lower[1]:
                if lower[2] is higher[2]:
                    upper_bound = max(lower[1], higher[1])
                    merged[-1] = (lower[0], upper_bound, lower[2])
                else:
                    if lower[1] > higher[1]:
                        merged[-1] = lower
                    else:
                        merged[-1] = (lower[0], higher[1], higher[2])
            else:
                merged.append(higher)
    return merged

In [6]:
def get_entities(df):

    entities = []

    for i in range(len(df)):
        entity = []

        for annot in df['annotation'][i]:
            try:
                ent = annot['label'][0]
                start = annot['points'][0]['start']
                end = annot['points'][0]['end'] + 1
                entity.append((start, end, ent))
            except:
                pass

        entity = mergeIntervals(entity)
        entities.append(entity)

    return entities

In [7]:
df_data['entities'] = get_entities(df_data)
df_data.head()

,content,annotation,entities
0,Abhishek Jha Application Development Associate...,"[{'label': ['Skills'], 'points': [{'start': 12...","[(0, 12, Name), (13, 46, Designation), (49, 58..."
1,Afreen Jamadar Active member of IIIT Committee...,"[{'label': ['Email Address'], 'points': [{'sta...","[(0, 14, Name), (62, 68, Location), (104, 148,..."
2,"Akhil Yadav Polemaina Hyderabad, Telangana - E...","[{'label': ['Skills'], 'points': [{'start': 37...","[(0, 21, Name), (22, 31, Location), (65, 117, ..."
3,Alok Khandai Operational Analyst (SQL DBA) Eng...,"[{'label': ['Skills'], 'points': [{'start': 80...","[(0, 12, Name), (13, 51, Designation), (54, 60..."
4,Ananya Chavan lecturer - oracle tutorials Mum...,"[{'label': ['Degree'], 'points': [{'start': 20...","[(0, 13, Name), (14, 22, Designation), (24, 41..."


In [8]:
def convert_dataturks_to_spacy(dataturks_JSON_FilePath):
    try:
        training_data = []
        lines = []
        with open(dataturks_JSON_FilePath, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            data = json.loads(line)
            text = data['content'].replace("\n", " ")
            entities = []
            data_annotations = data["annotation"]
            if data_annotations is not None:
                for annotation in data_annotations:

                    point = annotation['points'][0]
                    labels = annotation['label']

                    if not isinstance(labels, list):
                        labels = [labels]
                    
                    for label in labels:
                        point_start = point['start']
                        point_end = point['end']
                        point_text = point['text']
                        
                        lstrip_diff = len(point_text) - len(point_text.lstrip())
                        rstrip_diff = len(point_text) - len(point_text.rstrip())
                        if lstrip_diff != 0:
                            point_start = point_start + lstrip_diff
                        if rstrip_diff != 0:
                            point_end = point_end - rstrip_diff
                        entities.append((point_start, point_end + 1 , label))
            training_data.append((text, {"entities" : entities}))
        return training_data
    except Exception as e:
        logging.exception("Unable to process " + dataturks_JSON_FilePath + "\n" + "error = " + str(e))
        return None

def trim_entity_spans(data: list) -> list:
    """Removes leading and trailing white spaces from entity spans.

    Args:
        data (list): The data to be cleaned in spaCy JSON format.

    Returns:
        list: The cleaned data.
    """
    invalid_span_tokens = re.compile(r'\s')

    cleaned_data = []
    for text, annotations in data:
        entities = annotations['entities']
        valid_entities = []
        for start, end, label in entities:
            valid_start = start
            valid_end = end
            while valid_start < len(text) and invalid_span_tokens.match(
                    text[valid_start]):
                valid_start += 1
            while valid_end > 1 and invalid_span_tokens.match(
                    text[valid_end - 1]):
                valid_end -= 1
            valid_entities.append([valid_start, valid_end, label])
        cleaned_data.append([text, {'entities': valid_entities}])
    return cleaned_data 

In [9]:
data = trim_entity_spans(convert_dataturks_to_spacy("ner.json"))

In [10]:
import pandas as pd
from tqdm.notebook import tqdm

def clean_dataset(data):
    all_rows = []
    sum1 = 0
    
    for idx in tqdm(range(len(data))):
        start = 0
        
        num_words = len(data[idx][0].split())
        emptyList = ["Empty"] * num_words
        numberOfWords = 0
        
        lenOfString = len(data[idx][0])
        strData = data[idx][0]
        strDictData = data[idx][1]
        lastIndexOfSpace = strData.rfind(' ')
        
        for char_idx in range(lenOfString):
            
            if char_idx < lenOfString - 1:
                if (strData[char_idx] == " " and strData[char_idx+1] != " "):
                    for k, v in strDictData.items():
                        for j in range(len(v)):
                            entList = v[len(v)-j-1]
                            if (start >= int(entList[0]) and char_idx <= int(entList[1])):
                                emptyList[numberOfWords] = entList[2]
                                break
                            else:
                                continue
                    start = char_idx + 1  
                    numberOfWords += 1
                    
            if (char_idx == lastIndexOfSpace):
                 if strDictData:
                    last_v = list(strDictData.values())[-1]
                    for j in range(len(last_v)):
                        entList = last_v[len(last_v)-j-1]
                        if (lastIndexOfSpace >= int(entList[0]) and lenOfString <= int(entList[1])):
                            emptyList[numberOfWords] = entList[2]
                            numberOfWords += 1

        all_rows.append(emptyList)
        sum1 = sum1 + numberOfWords
        
    cleanedDF = pd.DataFrame({"sentences_cleaned": all_rows})
    
    return cleanedDF

In [11]:
cleanedDF = clean_dataset(data)

  0%|          | 0/220 [00:00<?, ?it/s]

In [12]:
cleanedDF.head()

,sentences_cleaned
0,"[Name, Name, Designation, Designation, Designa..."
1,"[Name, Name, Empty, Empty, Empty, Empty, Empty..."
2,"[Name, Name, Name, Empty, Empty, Empty, Empty,..."
3,"[Name, Name, Designation, Designation, Designa..."
4,"[Name, Name, Designation, Empty, Companies wor..."


In [13]:
unique_tags = set(cleanedDF['sentences_cleaned'].explode().unique())
tag2id = {tag: id for id, tag in enumerate(unique_tags)}
id2tag = {id : tag for tag, id in tag2id.items()}

In [14]:
unique_tags

{'College Name',
 'Companies worked at',
 'Degree',
 'Designation',
 'Email Address',
 'Empty',
 'Graduation Year',
 'Location',
 'Name',
 'Skills',
 'UNKNOWN',
 'Years of Experience'}

In [15]:
from keras.preprocessing.sequence import pad_sequences

MAX_LEN = 512
lables = cleanedDF['sentences_cleaned'].values.tolist()

tags = pad_sequences([[tag2id.get(l) for l in lab] for lab in lables],
                     maxlen=MAX_LEN, value=tag2id["Empty"], padding="post",
                     dtype="long", truncating="post")

In [16]:
tags

array([[10, 10,  7, ...,  9,  9,  9],
       [10, 10,  9, ...,  9,  9,  9],
       [10, 10, 10, ...,  9,  2,  9],
       ...,
       [10, 10,  7, ...,  9,  9,  9],
       [10, 10,  7, ...,  9,  9,  9],
       [10, 10,  7, ...,  9,  9,  9]], shape=(220, 512))

Tokenize and Align Labels with Library

In [17]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('tokenizer/')

In [18]:
label_all_token = True

def tokenize_and_align_labels(tokenizer, examples, tags):

    tokenized_inputs = tokenizer(examples, truncation=True, is_split_into_words=False, padding='max_length', max_length=512)
    labels = []

    for i, label in enumerate(tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        labels_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                labels_ids.append(-100)

            elif word_idx != previous_word_idx:
                labels_ids.append(label[word_idx])

            else:
                labels_ids.append(label[word_idx] if label_all_token else -100)
            previous_word_idx = word_idx

        labels.append(labels_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [19]:
test = tokenize_and_align_labels(tokenizer, df_data['content'].values.tolist(), tags)
train_dataset = tf.data.Dataset.from_tensor_slices((
    test['input_ids'],
    test['labels']
))

In [21]:
test['labels'][0]

[-100,
 np.int64(10),
 np.int64(10),
 np.int64(7),
 np.int64(7),
 np.int64(7),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(3),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(7),
 np.int64(7),
 np.int64(9),
 np.int64(0),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9),
 np.int64(9